Publication adaptation: workspace labels and runtime paths have been renamed. Saved scientific results and embedded image bytes are retained from the recorded run; this source notebook has not been recomputed after that adaptation. Printed historical checksums describe that recorded run. Current released-file checksums are in the publication source map.

# Core raw-panel visual and caption review
This QA-only notebook registers the exact files inspected individually by research, their SVG text/frame properties, and the reviewed caption bindings. It creates no plots and changes no analysis or manuscript. Human visual review uses the PNG companions; SVG geometry/text metadata provides a separate mechanical check, not a replacement for visual inspection.

Environment: existing **chronoclone**, existing Conda-compatible installation `/path/to/ChronoClone/.runtime/chronoclone`; kernel `chronoclone`. Versions and the exact execution command are printed below. This is a supplementary QA register, not a new scientific analysis.


In [1]:

from pathlib import Path
import sys,json,re,hashlib,platform,importlib.metadata as metadata,xml.etree.ElementTree as ET
import pandas as pd
from PIL import Image
from IPython.display import display
O=Path('/path/to/ChronoClone/analysis_workspace/singh/singh_reference_response_20260911/dynamics');ROOT=O.parents[3]
environment={'environment':'chronoclone','path':str(Path(sys.executable).parent.parent),'kernel':'chronoclone','Python':platform.python_version(),'versions':{p:metadata.version(p) for p in ['pandas','pillow','nbformat']},'execution_command':f'{sys.executable} {ROOT/"analysis_workspace/manuscript/chronoclone_panel_revision_20260910/execute_notebook.py"} {O/"0003_core_panel_visual_review.ipynb"}'}
assert environment['path']=='/path/to/ChronoClone/.runtime/chronoclone'
display(environment)


{'environment': 'chronoclone',
 'path': '/path/to/ChronoClone/.runtime/chronoclone',
 'kernel': 'chronoclone',
 'Python': '3.11.15',
 'versions': {'pandas': '2.3.3', 'pillow': '12.2.0', 'nbformat': '5.10.4'},
 'execution_command': '/path/to/ChronoClone/.runtime/chronoclone/bin/python /path/to/ChronoClone/analysis_workspace/manuscript/chronoclone_panel_revision_20260910/execute_notebook.py /path/to/ChronoClone/analysis_workspace/singh/singh_reference_response_20260911/dynamics/0003_core_panel_visual_review.ipynb'}

In [2]:

stems=['figure_5a_wt_reference_position','figure_5b_state_response','figure_5c_composition_response','figure_5d_mc3_state_contrasts','figure_5e_mc3_composition','figure_s6a_mc3_paired_families']
findings={
stems[0]:{'visual':'Pass after root correction: missing Day27 cells are light gray, labels and colorbar inside frame; no panel title.','caption':'Within-sample family means then equal observed-sample averaging correctly stated. Gray missing-support description now matches image.','resolved_issue':'Initial image had transparent/white missing cells; root changed colormap bad color to light gray and reran core notebook.'},
stems[1]:{'visual':'Pass: readable 17-row heatmap, four arm/day columns, visible star symbols, zero-centered scale, gray missing estimates.','caption':'Correctly distinguishes point support from inferential support; separate 68-test state BH, conditional family multiplier and star levels stated.'},
stems[2]:{'visual':'Pass: readable 17-row heatmap, gray missing estimates and visible symbols; sequential TVD scale 0–1.','caption':'Correctly states weighted within-sample TVD display, sample-by-category maximum-standardized contrast-vector test, and separate 68-test composition BH; no scalar TVD-null claim.'},
stems[3]:{'visual':'Pass: four estimates/intervals and two double-star labels, all labels inside frame; zero reference does not obscure marks.','caption':'Four contributing WT/mutant unique-family counts match effects.csv; pointwise intervals and full 68-test correction correctly distinguished.'},
stems[4]:{'visual':'Pass: three 100% stacked bars, large labels and full category legend inside exported frame; no panel prose.','caption':'Explicitly distinguishes equal-sample descriptive arm composition from minimum-arm-support weighted pairwise contrasts; pooling HSC+MPP to HSPC correctly stated.'},
stems[5]:{'visual':'Pass: six orange points below zero, readable numeric display indices 1–6 and axes; no clipping or prose.','caption':'Six matched families, at least two cells per arm/sample, one difference per family after averaging samples, raw P=.03125 and four-test q=.0625, and display indices versus barcodes correctly stated.'}}
manuscript=ROOT/'analysis_workspace/manuscript/chronoclone_v4_20260911/manuscript_source.md'
text=manuscript.read_text();ns={'s':'http://www.w3.org/2000/svg'};rows=[]
for stem in stems:
    png=O/'panels'/f'{stem}.png';svg=O/'panels'/f'{stem}.svg';node=ET.parse(svg).getroot();view=[float(v) for v in node.attrib['viewBox'].split()]
    texts=node.findall('.//s:text',ns);sizes=[];anchors=[]
    for label in texts:
        match=re.search(r'font-size:\s*([0-9.]+)px',label.attrib.get('style',''))
        if match:sizes.append(float(match.group(1)))
        if 'x' in label.attrib and 'y' in label.attrib:
            x=float(label.attrib['x']);y=float(label.attrib['y']);anchors.append(0<=x<=view[2] and 0<=y<=view[3])
    assert view[2]>0 and view[3]>0 and texts and all(anchors)
    rows.append({'stem':stem,'png_path':str(png.relative_to(ROOT)),'png_sha256':hashlib.sha256(png.read_bytes()).hexdigest(),'png_size_px':list(Image.open(png).size),'svg_path':str(svg.relative_to(ROOT)),'svg_sha256':hashlib.sha256(svg.read_bytes()).hexdigest(),'svg_viewbox':view,'editable_text_nodes':len(texts),'minimum_font_px':min(sizes) if sizes else None,'text_anchors_in_viewbox':all(anchors),'embedded_raster_images':len(node.findall('.//s:image',ns)),'individually_visually_inspected':True,'frame_and_readability':'pass','caption_binding':'pass',**findings[stem]})
# Verify the specific scope/weighting captions at this manuscript snapshot.
captions=[line for line in text.splitlines() if line.startswith('**Figure 5') or line.startswith('**Figure S6')]
core=__import__('nbformat').read(O/'0001_family_aware_responses.ipynb',as_version=4)
sources='\n'.join(c.source for c in core.cells if c.cell_type=='code')
assert "cmap=plt.get_cmap('viridis').copy();cmap.set_bad('#e6e6e6')" in sources
assert 'This descriptive arm summary uses equal sample weights' in text
assert 'they are not tests of scalar TVD' in text
assert 'numeric labels are display indices rather than original barcodes' in text
assert 'Each value is the smaller of the two contributing arm-specific unique-family counts.' in text
registry={'review_date':'2026-09-11','reviewer':'research independent dynamics subagent','scope':'Six raw PNG panels individually; SVG text/frame metadata; captions 5A–5E and S6A against producing code and saved tables. S6C minimum-arm support caption verified.','environment':environment,'panels':rows,'manuscript_path':str(manuscript.relative_to(ROOT)),'manuscript_sha256':hashlib.sha256(manuscript.read_bytes()).hexdigest(),'captions_at_review':captions,'limitations':['SVG text anchor check is not a glyph-bounding-box check; full-frame acceptance comes from individual PNG visual inspection.','Heatmap bodies and colorbars may contain embedded raster images; labels remain editable SVG text.','This register does not verify DOCX/Google page placement or claim independent experimental replication.'],'open_core_panel_defects':[]}
(O/'reports/core_panel_visual_review.json').write_text(json.dumps(registry,indent=2))
display(pd.DataFrame(rows)[['stem','png_sha256','editable_text_nodes','minimum_font_px','text_anchors_in_viewbox','embedded_raster_images','frame_and_readability']]);print(O/'reports/core_panel_visual_review.json')


,stem,png_sha256,editable_text_nodes,minimum_font_px,text_anchors_in_viewbox,embedded_raster_images,frame_and_readability
0,figure_5a_wt_reference_position,8b848ff28c31bc54a43708296b774355224bc993db4de5...,28,13.0,True,2,pass
1,figure_5b_state_response,ffcbee9be2fe34becda778bcb5a51164eb267a2d595efc...,46,13.0,True,2,pass
2,figure_5c_composition_response,d55b658274c80253218e48a9943108c29ca7e4939a3ac7...,51,13.0,True,2,pass
3,figure_5d_mc3_state_contrasts,cebef0f34d471bcea194de2563d43f1ca19cc46d2ea5ee...,12,13.0,True,0,pass
4,figure_5e_mc3_composition,d159e5030ab9ccd72f5c7c827810a03aab30bb3e3a09c3...,18,13.0,True,0,pass
5,figure_s6a_mc3_paired_families,4aee705b392908974e2815ee8a8d592aabe4fcaa051c90...,15,13.0,True,0,pass


/path/to/ChronoClone/analysis_workspace/singh/singh_reference_response_20260911/dynamics/reports/core_panel_visual_review.json
